In [ ]:
import sys,subprocess,time,datetime,json,hashlib,zipfile
from pathlib import Path
start=time.monotonic()
print(datetime.datetime.now(datetime.timezone.utc).isoformat(),"START HF write and recovery preflight",flush=True)
try:
    subprocess.run([sys.executable,"-m","pip","install","-q","huggingface_hub","kagglehub"],check=True)
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    import kagglehub
    secrets=UserSecretsClient()
    try:key=secrets.get_secret("HF_WRITE_TOKEN")
    except Exception:key=secrets.get_secret("HF_WRITE")
    api=HfApi(token=key)
    print("Authenticated HF account:",api.whoami()["name"],flush=True)
    repo="Finsafe-team/finsafe-recovery"
    api.create_repo(repo,repo_type="dataset",private=True,exist_ok=True)
    if not api.repo_info(repo,repo_type="dataset").private:raise RuntimeError("Recovery repository must remain private")
    path=kagglehub.notebook_output_download("keshavdixit11/lexsilabs/versions/5",path="finsafe-stage04b-recovery.zip")
    with zipfile.ZipFile(path) as z:
        if z.testzip():raise RuntimeError("Invalid recovery archive")
    digest=hashlib.sha256(Path(path).read_bytes()).hexdigest()
    if digest!="db8e2cbf4e40ad92ebf48b99ad7e3492ac5d60c03757d415db1d10d04c360ce0":raise RuntimeError("Recovery SHA mismatch")
    api.upload_file(path_or_fileobj=path,path_in_repo="bootstrap/version5-recovery.zip",repo_id=repo,repo_type="dataset")
    status={"status":"complete","archive_sha256":digest,"archive_bytes":Path(path).stat().st_size,"utc":datetime.datetime.now(datetime.timezone.utc).isoformat()}
    Path("/kaggle/working/hf-preflight.json").write_text(json.dumps(status,indent=2))
    api.upload_file(path_or_fileobj=json.dumps(status).encode(),path_in_repo="bootstrap/preflight.json",repo_id=repo,repo_type="dataset")
    print("Verified organization write access and private recovery backup:",json.dumps(status),flush=True)
except BaseException as error:
    print("FAILED preflight",type(error).__name__,flush=True)
    raise
else:print("DONE preflight",round(time.monotonic()-start,1),"seconds",flush=True)
